<a href="https://colab.research.google.com/github/jshingala/FlashAttention/blob/Tri/trition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CUDA available: True
Device: Tesla T4
Triton version: 3.6.0


In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def copy_kernel(
    x_ptr,            # pointer to input tensor in GPU memory
    out_ptr,          # pointer to output tensor in GPU memory
    n_elements,       # total number of elements
    BLOCK_SIZE: tl.constexpr,   # how many elements each program handles
):
    # 1. Which block am I? Programs are numbered 0, 1, 2, ...
    pid = tl.program_id(axis=0)

    # 2. Compute the range of indices THIS program is responsible for.
    #    If BLOCK_SIZE=1024 and pid=2, this covers indices 2048..3071.
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    # 3. Mask: the last block may run past the end of the tensor.
    #    This says "only touch indices that actually exist."
    mask = offsets < n_elements

    # 4. Load my block from global memory (masked), compute, store it back.
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x, mask=mask)


def copy(x: torch.Tensor):
    out = torch.empty_like(x)
    n_elements = x.numel()
    BLOCK_SIZE = 1024
    # The "grid" = how many programs to launch. We need enough blocks
    # to cover all elements: ceil(n_elements / BLOCK_SIZE).
    grid = (triton.cdiv(n_elements, BLOCK_SIZE),)
    copy_kernel[grid](x, out, n_elements, BLOCK_SIZE=BLOCK_SIZE)
    return out


# --- test it ---
x = torch.randn(10000, device="cuda")
out = copy(x)
print("Max difference from input:", (out - x).abs().max().item())
print("Match:", torch.allclose(out, x))

Max difference from input: 0.0
Match: True


In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def softmax_kernel(
    x_ptr,              # input  [n_rows, n_cols]
    out_ptr,            # output [n_rows, n_cols]
    row_stride,         # how many elements to skip to get to the next row
    n_cols,             # number of columns (real width of each row)
    BLOCK_SIZE: tl.constexpr,   # padded width (power of two >= n_cols)
):
    # 1. Which ROW am I responsible for?
    row_idx = tl.program_id(axis=0)

    # 2. Column offsets + pointers for this row.
    col_offsets = tl.arange(0, BLOCK_SIZE)
    ptrs = x_ptr + row_idx * row_stride + col_offsets

    # 3. Mask out the padding columns (BLOCK_SIZE may exceed n_cols).
    mask = col_offsets < n_cols

    # 4. Load the row; padding slots load -inf so exp(-inf)=0.
    row = tl.load(ptrs, mask=mask, other=float("-inf"))

    # 5. Stable softmax.
    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_out = numerator / denominator

    # 6. Store back to the same positions.
    out_ptrs = out_ptr + row_idx * row_stride + col_offsets
    tl.store(out_ptrs, softmax_out, mask=mask)


def softmax(x: torch.Tensor):
    n_rows, n_cols = x.shape
    # BLOCK_SIZE must be a power of two >= n_cols
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    # One program per row → grid is (n_rows,)
    grid = (n_rows,)
    softmax_kernel[grid](
        x, out,
        x.stride(0),     # row stride
        n_cols,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return out


# --- test against PyTorch ---
x = torch.randn(1823, 781, device="cuda")   # deliberately not round numbers
out_triton = softmax(x)
out_torch = torch.softmax(x, dim=-1)
print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-5))

Max difference: 7.450580596923828e-09
Match: True


In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def flash_attention_kernel(
    q_ptr, k_ptr, v_ptr, out_ptr,
    # strides for Q [B*H, N, D]
    stride_qh, stride_qm, stride_qd,
    stride_kh, stride_kn, stride_kd,
    stride_vh, stride_vn, stride_vd,
    stride_oh, stride_om, stride_od,
    N,                              # sequence length
    scale,                          # 1 / sqrt(head_dim)
    BLOCK_M: tl.constexpr,          # query rows per program
    BLOCK_N: tl.constexpr,          # key/value rows per inner step
    HEAD_DIM: tl.constexpr,         # head dimension D
):
    # Which query block, and which (batch*head) am I?
    pid_m = tl.program_id(axis=0)   # block-row of queries
    pid_h = tl.program_id(axis=1)   # which batch-head

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)   # query rows this program owns
    offs_d = tl.arange(0, HEAD_DIM)                     # the full head dimension

    # Pointers to this program's Q tile: [BLOCK_M, HEAD_DIM]
    q_ptrs = q_ptr + pid_h * stride_qh \
        + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)        # load once, reuse for all key blocks

    # Running softmax statistics (one per query row).
    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)   # running max
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)                 # running sum
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)        # running output

    # Walk the keys/values in blocks of BLOCK_N.
    for start_n in range(0, N, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)

        # Load K block [BLOCK_N, HEAD_DIM] and V block [BLOCK_N, HEAD_DIM].
        k_ptrs = k_ptr + pid_h * stride_kh \
            + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        v_ptrs = v_ptr + pid_h * stride_vh \
            + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        kv_mask = offs_n[:, None] < N
        k = tl.load(k_ptrs, mask=kv_mask, other=0.0)
        v = tl.load(v_ptrs, mask=kv_mask, other=0.0)

        # Scores for this block: Q @ Kᵀ  →  [BLOCK_M, BLOCK_N]
        s = tl.dot(q, tl.trans(k)) * scale
        # Mask out keys past the end of the sequence so they can't win the max
        # or contribute to the sum.
        s = tl.where(offs_n[None, :] < N, s, float("-inf"))

        # ---- the online-softmax recurrence ----
        m_new = tl.maximum(m_i, tl.max(s, axis=1))     # extend the running max
        alpha = tl.exp(m_i - m_new)                    # correction factor for old state
        p = tl.exp(s - m_new[:, None])                 # this block's weights, stable

        l_i = alpha * l_i + tl.sum(p, axis=1)          # rescale old sum, add new
        acc = alpha[:, None] * acc + tl.dot(p, v)      # rescale old output, add new block
        m_i = m_new

    # Normalize exactly once, after all blocks.
    acc = acc / l_i[:, None]

    # Store the [BLOCK_M, HEAD_DIM] output tile.
    out_ptrs = out_ptr + pid_h * stride_oh \
        + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(out_ptrs, acc, mask=q_mask)


def flash_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
    # q, k, v: [B, H, N, D]
    B, H, N, D = q.shape
    # Flatten batch and head into one axis: [B*H, N, D]
    q = q.reshape(B * H, N, D)
    k = k.reshape(B * H, N, D)
    v = v.reshape(B * H, N, D)
    out = torch.empty_like(q)

    scale = 1.0 / (D ** 0.5)
    BLOCK_M, BLOCK_N = 64, 64
    grid = (triton.cdiv(N, BLOCK_M), B * H)

    flash_attention_kernel[grid](
        q, k, v, out,
        q.stride(0), q.stride(1), q.stride(2),
        k.stride(0), k.stride(1), k.stride(2),
        v.stride(0), v.stride(1), v.stride(2),
        out.stride(0), out.stride(1), out.stride(2),
        N,
        scale,
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, HEAD_DIM=D,
    )
    return out.reshape(B, H, N, D)


# --- test against PyTorch ---
torch.manual_seed(0)
B, H, N, D = 2, 4, 512, 64
q = torch.randn(B, H, N, D, device="cuda")
k = torch.randn(B, H, N, D, device="cuda")
v = torch.randn(B, H, N, D, device="cuda")

out_triton = flash_attention(q, k, v)

# Reference: plain softmax(Q·Kᵀ / sqrt(D)) · V
scores = (q @ k.transpose(-2, -1)) / (D ** 0.5)
out_torch = torch.softmax(scores, dim=-1) @ v

print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-2))

Max difference: 4.76837158203125e-07
Match: True
